# Paso 2 · Dataset limpio de Reef Check (opción A)

**Decisión del paso 1:** entrenar solo con Reef Check, un único protocolo de muestreo con escala coherente.

Este notebook crea `data/coral_reefcheck.csv` desde el CSV original de Kaggle. Cosas que corregimos respecto a la limpieza anterior:

1. **Filas duplicadas:** cada muestreo de Reef Check aparece 2 veces (una fila por tipo de sustrato: coral duro y algas). Pasamos a **una fila por muestreo** y convertimos la cobertura en dos variables útiles.
2. **Sin imputación:** faltan <0,5 % de valores; los eliminamos en lugar de inventarlos.
3. **Conservamos `Site_ID` y la fecha** para hacer una partición de entrenamiento/prueba honesta (paso 3).

In [1]:
import kagglehub, pandas as pd, numpy as np

path = kagglehub.dataset_download("mehrdat/coral-reef-global-bleaching")
raw = pd.read_csv(f"{path}/coral.csv", low_memory=False).replace("nd", np.nan)

rc = raw[(raw["Data_Source"] == "Reef_Check") & raw["Percent_Bleaching"].notna()].copy()
print(len(rc), "filas ·", rc["Sample_ID"].nunique(), "muestreos ·", rc["Site_ID"].nunique(), "sitios")

/Users/veronica/Desktop/Bootcamp/finalproject/corals/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


22531 filas · 11297 muestreos · 3920 sitios


## 1. ¿Por qué hay 2 filas por muestreo?
Solo cambian `Substrate_Name` y `Percent_Cover`. El blanqueamiento es el mismo.

In [2]:
dup_id = rc.loc[rc["Sample_ID"].duplicated(), "Sample_ID"].iloc[0]
rc.loc[rc["Sample_ID"] == dup_id, ["Sample_ID", "Substrate_Name", "Percent_Cover", "Percent_Bleaching", "TSA_DHW"]]

,Sample_ID,Substrate_Name,Percent_Cover,Percent_Bleaching,TSA_DHW
6981,10313996,Hard Coral,29.38,0,0
6982,10313996,Nutrient Indicator Algae,0.62,0,0


## 2. Una fila por muestreo + cobertura como variables
- `hard_coral_cover`: % del fondo cubierto por coral vivo. Importa porque el blanqueamiento se mide sobre el coral que hay.
- `algae_cover`: algas indicadoras de nutrientes. Señal de un arrecife degradado o con contaminación.

In [3]:
rc["Percent_Cover"] = pd.to_numeric(rc["Percent_Cover"], errors="coerce")
cover = (rc.pivot_table(index="Sample_ID", columns="Substrate_Name", values="Percent_Cover", aggfunc="first")
           .rename(columns={"Hard Coral": "hard_coral_cover", "Nutrient Indicator Algae": "algae_cover"}))
cover = cover[[c for c in ["hard_coral_cover", "algae_cover"] if c in cover.columns]]

surveys = rc.drop(columns=["Substrate_Name", "Percent_Cover"]).drop_duplicates("Sample_ID").set_index("Sample_ID")
surveys = surveys.join(cover).reset_index()
print(len(surveys), "muestreos")

11297 muestreos


## 3. Seleccionar y renombrar columnas

**Temperaturas:** vienen en Kelvin → las pasamos a °C.

**Glosario rápido** (todas las métricas térmicas vienen del satélite, base de datos CoRTAD, en la semana del muestreo):
- `sst`: temperatura del mar esa semana.
- `clim_sst`: temperatura media habitual del sitio.
- `ssta`: anomalía respecto a la media de *esa semana del año*.
- `tsa`: anomalía respecto al *máximo habitual de verano* del sitio. **La que importa al coral.**
- `tsa_dhw`: estrés térmico acumulado (°C·semanas) por encima del máximo de verano en las últimas 12 semanas. **Equivale al DHW de NOAA.**
- `tsa_frequency`: cuántas veces en el último año se superó el máximo de verano en ≥1 °C (estrés repetido).
- `ssta_sd`: cuánto varía la temperatura en el sitio. Los corales de aguas variables suelen ser más resistentes.
- `tsa_dhw_max`: el mayor DHW de **toda la serie del sitio**. ⚠️ Incluye años posteriores al muestreo, así que **no la usaremos como predictor** (sería mirar el futuro).

In [4]:
K = 273.15
df = pd.DataFrame({
    "sample_id": surveys["Sample_ID"],
    "site_id": surveys["Site_ID"],
    "date": pd.to_datetime(surveys["Date"]),
    "latitude": pd.to_numeric(surveys["Latitude_Degrees"]),
    "longitude": pd.to_numeric(surveys["Longitude_Degrees"]),
    "ocean": surveys["Ocean_Name"],
    "realm": surveys["Realm_Name"],
    "ecoregion": surveys["Ecoregion_Name"],
    "exposure": surveys["Exposure"],
})
num = {
    "distance_to_shore": "Distance_to_Shore", "turbidity": "Turbidity", "cyclone_frequency": "Cyclone_Frequency",
    "depth_m": "Depth_m", "windspeed": "Windspeed", "percent_bleaching": "Percent_Bleaching",
    "ssta": "SSTA", "ssta_frequency": "SSTA_Frequency", "ssta_dhw": "SSTA_DHW", "ssta_sd": "SSTA_Standard_Deviation",
    "tsa": "TSA", "tsa_frequency": "TSA_Frequency", "tsa_dhw": "TSA_DHW", "tsa_dhw_max": "TSA_DHWMax",
    "sst_k": "Temperature_Kelvin", "sst_max_k": "Temperature_Maximum", "clim_sst_k": "ClimSST",
    "sst_sd": "Temperature_Kelvin_Standard_Deviation",
}
for new, old in num.items():
    df[new] = pd.to_numeric(surveys[old], errors="coerce").values
df["hard_coral_cover"] = surveys["hard_coral_cover"].values
df["algae_cover"] = surveys["algae_cover"].values if "algae_cover" in surveys else np.nan

for c in ["sst", "sst_max", "clim_sst"]:
    df[c] = df.pop(f"{c}_k") - K
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["abs_latitude"] = df["latitude"].abs()
df.describe().T.round(2)

,count,mean,min,25%,50%,75%,max,std
sample_id,11297.0,10314851.718155,10307600.0,10311209.0,10314702.0,10318650.0,10322068.0,4199.143409
site_id,11297.0,5420.481455,4.0,3313.0,5536.0,7190.0,15736.0,3365.198843
date,11297,2011-04-28 14:37:29.163494400,1998-08-23 00:00:00,2006-09-12 00:00:00,2011-03-19 00:00:00,2015-10-28 00:00:00,2020-08-15 00:00:00,NaN
latitude,11297.0,3.915961,-28.8645,-8.6767,5.8144,15.9786,34.0981,15.231662
longitude,11297.0,61.604912,-179.8594,-35.0852,105.6672,123.5746,179.9645,97.265472
distance_to_shore,11296.0,2839.62983,3.68,97.68,253.65,997.4775,150705.93,10882.140409
turbidity,11295.0,0.078661,0.0,0.0399,0.0592,0.0842,1.1635,0.066544
cyclone_frequency,11297.0,51.248638,19.86,47.91,49.88,54.44,92.3,6.341979
depth_m,11297.0,6.513862,0.1,4.0,6.0,10.0,23.0,3.397375
windspeed,11244.0,4.679651,0.0,3.0,5.0,6.0,14.0,2.176424


## 4. Valores de relleno
`clim_sst` vale exactamente −11 °C (262,15 K) en miles de filas. No es una temperatura real: es un **valor de relleno para "sin dato"** en el dataset original. Lo convertimos en dato faltante. No necesitamos esta columna para el modelo (ya tenemos `tsa`, que mide lo mismo de forma más útil), así que no eliminamos esas filas.

In [5]:
print("clim_sst = −11 °C (relleno):", (df["clim_sst"].round(2) == -11.0).sum())
df.loc[df["clim_sst"].round(2) == -11.0, "clim_sst"] = np.nan
print("sst < 15 °C:", (df["sst"] < 15).sum())

clim_sst = −11 °C (relleno): 1831
sst < 15 °C: 0


In [6]:
before = len(df)
df = df.dropna(subset=["sst", "ssta", "tsa", "tsa_dhw", "windspeed", "hard_coral_cover"])
print(f"{before} → {len(df)} muestreos ({before - len(df)} eliminados)")

11297 → 11179 muestreos (118 eliminados)


## 5. Comprobaciones rápidas
- ¿Sube el blanqueamiento con `tsa_dhw`? (debería)
- ¿Cuántos muestreos hay con estrés alto? (pocos: es la limitación principal)
- ¿Cuántas veces se visita cada sitio? (muchas → en el paso 3 separaremos train/test **por sitio y por tiempo**, no al azar)

In [7]:
bins = pd.cut(df["tsa_dhw"], [-0.1, 0, 2, 4, 8, 100], labels=["0", "0–2", "2–4", "4–8", "8+"])
display(df.groupby(bins, observed=False)["percent_bleaching"].agg(["size", "mean", "median"]).round(1))
visits = df.groupby("site_id").size()
print(f"Sitios: {visits.size} · visitas por sitio: mediana {visits.median():.0f}, máx {visits.max()}")
print("Muestreos por año:", df["year"].value_counts().sort_index().to_dict())

,size,mean,median
tsa_dhw,,,
0,8261,1.8,0.0
0–2,1468,2.8,0.0
2–4,842,5.5,0.0
4–8,429,7.1,0.0
8+,179,8.6,0.0


Sitios: 3897 · visitas por sitio: mediana 2, máx 45
Muestreos por año: {1998: 5, 2000: 2, 2001: 3, 2002: 62, 2003: 609, 2004: 739, 2005: 688, 2006: 853, 2007: 651, 2008: 696, 2009: 702, 2010: 543, 2011: 543, 2012: 569, 2013: 641, 2014: 609, 2015: 630, 2016: 636, 2017: 791, 2018: 621, 2019: 586}


In [8]:
df.to_csv("../data/coral_reefcheck.csv", index=False)
print("Guardado:", df.shape)

Guardado: (11179, 32)
